# 02 - Gold: Contratos (SECOP enriquecido con RUES y TRM)

Trabajo Práctico 2 - Transformación Silver → Gold

## Objetivo

Construir `gold.fact_contratos` a partir de `silver.contratos_secop`, enriquecido con:

* `gold.dim_empresa` (debe existir — lo crea `01_Gold_Empresas.ipynb`) vía `documento_proveedor = numero_identificacion`, para traer `organizacion_juridica` y `cod_ciiu_act_econ_pri` del proveedor cuando está registrado en RUES.
* `silver.tasa_cambio` promedio mensual, para convertir `valor_del_contrato` a USD cuando `fecha_de_firma` cae dentro de la ventana de la TRM.

Grano: una fila por contrato (`id_contrato`).

## Preguntas de negocio que responde

| # | Pregunta |
|---|---|
| 1 | ¿Qué entidades, sectores y departamentos concentran más valor contratado? |
| 2 | ¿Qué tanto se contrata de forma directa? ¿Cambia por sector o entidad? |
| 3 | ¿Qué participación tienen las pymes en número de contratos frente a valor contratado? |
| 4 | ¿Qué tan ejecutados están los contratos? (`valor_pagado / valor_del_contrato`, y `dias_adicionados` como señal de prórroga) |
| 5 | ¿Qué proveedores concentran más contratos o más valor? |
| 10 | ¿Cuánto vale la contratación en dólares por mes? (solo contratos firmados dentro de la ventana de la TRM) |

Todas se responden agrupando esta única tabla por distintas columnas — no se necesita una tabla Gold por pregunta.

## Advertencia de cobertura (verificada en vivo antes de escribir este notebook)

El join `documento_proveedor = numero_identificacion` solo encuentra empresa en RUES para el **15.6%** de los contratos (17.996 de 115.679). No es un bug: nuestro RUES Bronze no cubre el 100% de las cámaras de comercio del país, y muchos proveedores de SECOP son personas naturales con cédula, entidades extranjeras o consorcios sin NIT propio. Las columnas `organizacion_juridica_proveedor` y `cod_ciiu_proveedor` quedan `NULL` para el 84.4% restante — visible y trazable, no se descartan contratos por esto. (Esta misma limitación es la razón por la que las preguntas 11 y 12 del negocio quedaron fuera de alcance: necesitan un RUES completo.)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_contratos = spark.table("Datos_Empresas.silver.contratos_secop")
df_dim_empresa = spark.table("Datos_Empresas.gold.dim_empresa")
df_trm = spark.table("Datos_Empresas.silver.tasa_cambio")

print(f"silver.contratos_secop: {df_contratos.count():,} filas")
print(f"gold.dim_empresa: {df_dim_empresa.count():,} filas")
print(f"silver.tasa_cambio: {df_trm.count():,} filas")

---

## Construcción de `gold.fact_contratos`

1. Join `LEFT` con `dim_empresa` por `documento_proveedor = numero_identificacion`.
2. TRM promedio mensual, unida por el mes de `fecha_de_firma`, para derivar `valor_contrato_usd`.
3. Indicador de ejecución `pct_ejecutado = valor_pagado / valor_del_contrato`.

In [ ]:
# numero_identificacion NO es clave unica en dim_empresa: 57.108 identificaciones
# tienen matricula en mas de una camara de comercio (legitimo, no es un rechazo).
# Si se une SECOP contra dim_empresa sin deduplicar, cada contrato se multiplica
# por el numero de matriculas de su proveedor (fan-out). Se deduplica aqui, quedandonos
# con la matricula mas reciente (mayor fecha_actualizacion) por numero_identificacion,
# igual que el criterio de "version mas reciente" ya usado en Silver para duplicados.
w_identificacion = Window.partitionBy("numero_identificacion").orderBy(F.col("fecha_actualizacion").desc_nulls_last())

df_empresa_proveedor = (
    df_dim_empresa
    .withColumn("_rank_identificacion", F.row_number().over(w_identificacion))
    .filter(F.col("_rank_identificacion") == 1)
    .select(
        F.col("numero_identificacion"),
        F.col("organizacion_juridica").alias("organizacion_juridica_proveedor"),
        F.col("cod_ciiu_act_econ_pri").alias("cod_ciiu_proveedor"),
    )
)

df_trm_mensual = (
    df_trm.withColumn("mes", F.date_trunc("month", F.col("vigenciadesde")))
    .groupBy("mes")
    .agg(F.round(F.avg("valor"), 2).alias("trm_promedio"))
)

df_fact_contratos = (
    df_contratos
    .join(df_empresa_proveedor, df_contratos["documento_proveedor"] == df_empresa_proveedor["numero_identificacion"], how="left")
    .drop("numero_identificacion")
    .withColumn("pct_ejecutado", F.round(F.col("valor_pagado") / F.col("valor_del_contrato"), 4))
    .withColumn("mes_firma", F.date_trunc("month", F.col("fecha_de_firma")))
    .join(df_trm_mensual, F.col("mes_firma") == df_trm_mensual["mes"], how="left")
    .withColumn("valor_contrato_usd", F.round(F.col("valor_del_contrato") / F.col("trm_promedio"), 2))
    .drop("mes", "mes_firma")
    .withColumn("_processed_at", F.current_timestamp())
)

total_contratos = df_contratos.count()
total_fact = df_fact_contratos.count()
print(f"silver.contratos_secop: {total_contratos:,} | gold.fact_contratos: {total_fact:,}")
assert total_contratos == total_fact, "El join duplico o perdio contratos (fan-out) -- revisar deduplicacion del proveedor"

df_fact_contratos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "Datos_Empresas.gold.fact_contratos"
)
print(f"Escrito: Datos_Empresas.gold.fact_contratos ({total_fact:,} filas)")

---

## Validación

In [ ]:
%sql
-- Q1: valor contratado por sector (top 10)
SELECT sector, COUNT(*) AS contratos, ROUND(SUM(valor_del_contrato), 0) AS valor_total
FROM Datos_Empresas.gold.fact_contratos
GROUP BY sector
ORDER BY valor_total DESC
LIMIT 10;

In [ ]:
%sql
-- Q2: % de contratacion directa global y por sector
SELECT sector,
  COUNT(*) AS contratos,
  SUM(CASE WHEN modalidad_de_contratacion = 'Contratación directa' THEN 1 ELSE 0 END) AS directos,
  ROUND(100.0 * SUM(CASE WHEN modalidad_de_contratacion = 'Contratación directa' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_directa
FROM Datos_Empresas.gold.fact_contratos
GROUP BY sector
ORDER BY contratos DESC
LIMIT 10;

In [ ]:
%sql
-- Q3: pymes, contratos vs valor
SELECT es_pyme,
  COUNT(*) AS contratos,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_contratos,
  ROUND(SUM(valor_del_contrato), 0) AS valor_total,
  ROUND(100.0 * SUM(valor_del_contrato) / SUM(SUM(valor_del_contrato)) OVER (), 1) AS pct_valor
FROM Datos_Empresas.gold.fact_contratos
GROUP BY es_pyme;

In [ ]:
%sql
-- Q4: ejecucion (pct_ejecutado) y prorrogas (dias_adicionados > 0)
SELECT
  ROUND(AVG(pct_ejecutado), 4) AS pct_ejecutado_promedio,
  SUM(CASE WHEN dias_adicionados > 0 THEN 1 ELSE 0 END) AS contratos_con_prorroga,
  COUNT(*) AS total_contratos
FROM Datos_Empresas.gold.fact_contratos;

In [ ]:
%sql
-- Q5: concentracion de proveedores (top 10 por valor)
SELECT proveedor_adjudicado, COUNT(*) AS contratos, ROUND(SUM(valor_del_contrato), 0) AS valor_total
FROM Datos_Empresas.gold.fact_contratos
GROUP BY proveedor_adjudicado
ORDER BY valor_total DESC
LIMIT 10;

In [ ]:
%sql
-- Q10: valor contratado en USD por mes (solo dentro de la ventana TRM)
SELECT date_trunc('month', fecha_de_firma) AS mes,
  COUNT(*) AS contratos,
  ROUND(SUM(valor_contrato_usd), 0) AS valor_usd
FROM Datos_Empresas.gold.fact_contratos
WHERE valor_contrato_usd IS NOT NULL
GROUP BY date_trunc('month', fecha_de_firma)
ORDER BY mes;

---

## Siguiente paso

Con Silver, DQL y Gold completos, continuar con la orquestación (`Trabajo Practico 2 - Orquestacion/`) y la prueba de idempotencia.